# Analysis of climate impact on house prices

In [1]:
import sys
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import statsmodels as sm
import statsmodels.formula.api as smf
from patsy import dmatrices
import re
from scipy import stats
import geopandas as gpd
import matplotlib.patches as mpatches
from pathlib import Path
import warnings

In [2]:
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f"{x:,.4f}")
sns.set_theme(style='whitegrid', context='talk')

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

SRC_DIR = ROOT / 'src'
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

ECONOMIC_DIR = ROOT / 'data' / 'economic'
POPULATION_DIR = ROOT / 'data' / 'population'
CACHE_DIR = ROOT / 'data' / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

from polars_cached_io import (
    read_cew_employment_wages_cached,
    read_cew_totals_cached,
    read_fema_disasters_cached,
    read_redfin_county_cached,
)

LAST_COMPLETE_YEAR = pd.Timestamp.today().year - 1
TARGET_YEARS = list(range(LAST_COMPLETE_YEAR - 9, LAST_COMPLETE_YEAR + 1))
TARGET_YEARS

[2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

## Load and Transform Data

In [3]:
# Load data into pd DataFrames. Large CSVs are projected and cached through Polars before conversion back to pandas.
natural_disasters_df = read_fema_disasters_cached(
    ROOT / 'data' / 'climate' / 'FEMA_Disaster_Declarations.csv',
    cache_dir=CACHE_DIR,
)
nri_county_df = pd.read_csv(ROOT / 'data' / 'climate' / 'NRI_Table_Counties.csv')
housing_county_df = read_redfin_county_cached(
    ROOT / 'data' / 'housing' / 'Redfin-Housing-Market-By-County.csv',
    cache_dir=CACHE_DIR,
)
fips_df = pd.read_csv(ROOT / 'data' / 'geographic' / 'fips_master_v2.csv')
personal_income_df = pd.read_csv(
    ROOT / 'data' / 'economic' / 'BEA - US, States, Counties - Personal Income.csv',
    usecols=["IBRC_GEO_ID", "Year", "Data", "Linecode Description"],
    low_memory=False,
)

nri_county_df = (
    nri_county_df.assign(
        fips = nri_county_df["STCOFIPS"].astype(str).str.zfill(5),
        nri_risk_score = pd.to_numeric(nri_county_df["RISK_SCORE"], errors = "coerce"),
        nri_risk_rating = nri_county_df["RISK_RATNG"],
        nri_risk_rating_date = nri_county_df["NRI_VER"]
    )[["fips", "nri_risk_score", "nri_risk_rating", "nri_risk_rating_date"]]
    .drop_duplicates(subset = ["fips"])
)

### Economic and Demographic data


#### Paths

The notebook reads the exact files requested in the task. The economic filename in the prompt contains a typo; the file on disk is `CEW - US, States, Counties - Total Ownership.csv`.


In [4]:

bea_income_path = ECONOMIC_DIR / 'BEA - US, States, Counties - Personal Income.csv'
cew_total_path = ECONOMIC_DIR / 'CEW - US, States, Counties - Total Ownership.csv'
pop_change_path = POPULATION_DIR / 'Components of Population Change - U.S., States, and Counties.csv'
pop_age_sex_path = POPULATION_DIR / 'Population by Age and Sex - US, States, Counties.csv'
pop_race_path = POPULATION_DIR / 'Population by Race - US, States, Counties.csv'
pop_estimates_path = POPULATION_DIR / 'Population Estimates - U.S., States, and Counties.csv'

for path in [
    bea_income_path,
    cew_total_path,
    pop_change_path,
    pop_age_sex_path,
    pop_race_path,
    pop_estimates_path,
]:
    print(path.relative_to(ROOT))


data\economic\BEA - US, States, Counties - Personal Income.csv
data\economic\CEW - US, States, Counties - Total Ownership.csv
data\population\Components of Population Change - U.S., States, and Counties.csv
data\population\Population by Age and Sex - US, States, Counties.csv
data\population\Population by Race - US, States, Counties.csv
data\population\Population Estimates - U.S., States, and Counties.csv


In [5]:
from cluster_county_econ_demographics import BEA_LINECODES


def add_county_fips(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['Statefips'] = pd.to_numeric(out['Statefips'], errors='coerce')
    out['Countyfips'] = pd.to_numeric(out['Countyfips'], errors='coerce')
    out = out.dropna(subset=['Statefips', 'Countyfips'])
    out['state_fips'] = out['Statefips'].astype(int).astype(str).str.zfill(2)
    out['county_fips'] = out['Countyfips'].astype(int).astype(str).str.zfill(3)
    out = out[(out['state_fips'] != '00') & (out['county_fips'] != '000')].copy()
    out['fips'] = out['state_fips'] + out['county_fips']
    return out


def dedupe_by_key(df: pd.DataFrame, key_cols: list[str], strategy: str = 'first') -> pd.DataFrame:
    if not df.duplicated(key_cols).any():
        return df

    if strategy == 'largest_magnitude':
        numeric_cols = [
            col for col in df.columns
            if col not in key_cols and pd.api.types.is_numeric_dtype(df[col])
        ]
        score = df[numeric_cols].fillna(0).abs().sum(axis=1)
        return (
            df.assign(_score=score)
              .sort_values(key_cols + ['_score'], ascending=[True] * len(key_cols) + [False])
              .drop_duplicates(key_cols)
              .drop(columns='_score')
        )

    if strategy == 'count_over_estimate':
        rank = df['Count or Estimate'].map({'Count': 0, 'Estimate': 1}).fillna(2)
        return (
            df.assign(_rank=rank)
              .sort_values(key_cols + ['_rank'])
              .drop_duplicates(key_cols)
              .drop(columns='_rank')
        )

    return df.sort_values(key_cols).drop_duplicates(key_cols)


def read_cew_totals(path: Path, chunksize: int | None = None) -> pd.DataFrame:
    return read_cew_totals_cached(path, cache_dir=CACHE_DIR)

#### Read the Requested CSV Files Into Individual DataFrames

In [6]:

bea_income_raw = pd.read_csv(
    bea_income_path,
    usecols=['Statefips', 'Countyfips', 'Description', 'Year', 'Linecode', 'Linecode Description', 'Data'],
    low_memory=False,
)
bea_income_df = add_county_fips(bea_income_raw)
bea_income_df = bea_income_df[bea_income_df['Linecode'].isin(BEA_LINECODES)].copy()
bea_income_df['metric'] = bea_income_df['Linecode'].map(BEA_LINECODES)

cew_total_df = read_cew_totals(cew_total_path)

population_change_df = pd.read_csv(pop_change_path, low_memory=False)
population_change_df = add_county_fips(population_change_df)
population_change_df = dedupe_by_key(population_change_df, ['fips', 'Year'], strategy='largest_magnitude')

population_age_sex_df = pd.read_csv(pop_age_sex_path, low_memory=False)
population_age_sex_df = add_county_fips(population_age_sex_df)
population_age_sex_df = dedupe_by_key(population_age_sex_df, ['fips', 'Year'])

population_race_df = pd.read_csv(pop_race_path, low_memory=False)
population_race_df = add_county_fips(population_race_df)
population_race_df = dedupe_by_key(population_race_df, ['fips', 'Year'])

population_estimates_df = pd.read_csv(pop_estimates_path, low_memory=False)
population_estimates_df = add_county_fips(population_estimates_df)
population_estimates_df = population_estimates_df[
    population_estimates_df['State or County Release'].eq('County')
].copy()
population_estimates_df = dedupe_by_key(
    population_estimates_df,
    ['fips', 'Year'],
    strategy='count_over_estimate',
)

for name, df in {
    'bea_income_df': bea_income_df,
    'cew_total_df': cew_total_df,
    'population_change_df': population_change_df,
    'population_age_sex_df': population_age_sex_df,
    'population_race_df': population_race_df,
    'population_estimates_df': population_estimates_df,
}.items():
    print(f"{name:24s} rows={len(df):>9,} counties={df['fips'].nunique():>5,} years={int(df['Year'].min())}-{int(df['Year'].max())}")


bea_income_df            rows=  339,365 counties=1,406 years=2001-2023
cew_total_df             rows=   34,283 counties=1,444 years=2001-2024
population_change_df     rows=  113,950 counties=3,158 years=1981-2025
population_age_sex_df    rows=   78,575 counties=3,156 years=2000-2024
population_race_df       rows=  109,970 counties=3,158 years=1990-2024
population_estimates_df  rows=   91,145 counties=3,158 years=1970-2025


### FIPS code data

#### Cleaning and Preprocessing

In [7]:
# For county FIPS codes

# Strip trailing and leading whitespaces from all values
fips_df = fips_df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)

# Convert fips from int to 5-digit str, padded with zeroes to the left
fips_df['fips'] = (
    fips_df['fips']
        .astype('str')
        .str.zfill(5)
)

# Hawaii's county master combines Maui and Kalawao under 15009, but FEMA emits
# a separate 15005 county code for Kalawao. Add a county-level alias from the
# existing Hawaii reference row so disaster joins can resolve both codes.
kalawao_alias = fips_df.loc[fips_df['fips'] == '15009'].copy()
kalawao_alias['fips'] = '15005'
kalawao_alias['county_name'] = 'Kalawao County'
fips_df = pd.concat([fips_df, kalawao_alias], ignore_index=True)

fips_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3219 entries, 0 to 3218
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   fips         3219 non-null   str  
 1   county_name  3168 non-null   str  
 2   state        3219 non-null   str  
 3   state_long   3219 non-null   str  
 4   msa_code     1808 non-null   str  
 5   msa_name     1808 non-null   str  
 6   msa_type     1808 non-null   str  
 7   csa_code     1150 non-null   str  
 8   csa_name     1150 non-null   str  
dtypes: str(9)
memory usage: 412.4 KB


### Natural disasters data

In [8]:
# Types of natural disasters
natural_disasters_df["incidentType"].unique()

<ArrowStringArray>
[               'Fire',        'Severe Storm', 'Straight-Line Winds',
               'Flood',           'Hurricane',          'Biological',
        'Winter Storm',             'Tornado',      'Tropical Storm',
          'Earthquake',             'Typhoon',           'Snowstorm',
            'Freezing',       'Mud/Landslide',       'Coastal Storm',
               'Other',    'Severe Ice Storm',     'Dam/Levee Break',
   'Volcanic Eruption', 'Tropical Depression',    'Toxic Substances',
            'Chemical',           'Terrorist',             'Drought',
         'Human Cause',      'Fishing Losses',             'Tsunami']
Length: 27, dtype: str

#### Cleaning and Preprocessing

1. Prepare full 5-digit FIPS code from fipsStateCode and fipsCountyCode
2. Join natural disasters data with county FIPS data to get the counties where disasters were declared  
Note: State-wide disasters should match on the state FIPS (XX000) from the FIPS data

In [9]:
# Strip trailing and leading whitespaces from all values
natural_disasters_df = natural_disasters_df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)

# Convert fipsStateCode into a 2-digit string, fipsCountyCode into a 3-digit string
# then concatenate them to build the 5‑digit FIPS
natural_disasters_df['fipsStateCode'] = (
    natural_disasters_df['fipsStateCode']
        .astype('str')
        .str.zfill(2)
)
natural_disasters_df['fipsCountyCode'] = (
    natural_disasters_df['fipsCountyCode']
        .astype('str')
        .str.zfill(3)
)
natural_disasters_df['fips_code_full'] = (
    natural_disasters_df['fipsStateCode'] + natural_disasters_df['fipsCountyCode']
)

# Change columns with date values to datetime type
natural_disasters_df["declarationDate"] = pd.to_datetime(natural_disasters_df["declarationDate"])
natural_disasters_df["incidentBeginDate"] = pd.to_datetime(natural_disasters_df["incidentBeginDate"])
natural_disasters_df["incidentEndDate"] = pd.to_datetime(natural_disasters_df["incidentEndDate"])

# Join natural_disasters_df on fips_df
natural_disasters_df = pd.merge(
    left=natural_disasters_df,
    right=fips_df,
    how='left',
    left_on='fips_code_full',
    right_on='fips'
)

# Drop duplicate columns, rename columns
natural_disasters_df.drop(columns = ["state_x", "fips_code_full"], inplace = True)
natural_disasters_df = natural_disasters_df.rename(columns = {
    "state_y": "state"
})

Notes on results from joining natural disaster data on FIPS code data:

_merge<br>
both          67082<br>
left_only      2464<br>
right_only        0<br>
Name: count, dtype: int64<br>

The natural disasters that failed to join on fips_df are associated with states: ['MP', 'GU', 'AS', 'PR', 'VI', 'FM', 'MH', 'PW']<br>
Their fipsStateCode / fipsCountyCode do not match those from the fips code master dataset.

In [10]:
# Select relevant columns from natural_disasters_df
natural_disasters_df = natural_disasters_df[["fips",
                                             "county_name",
                                             "state",
                                             "disasterNumber",
                                             "declarationDate", 
                                             "incidentType", 
                                             "declarationTitle", 
                                             "ihProgramDeclared", 
                                             "iaProgramDeclared", 
                                             "paProgramDeclared", 
                                             "hmProgramDeclared", 
                                             "incidentBeginDate", 
                                             "incidentEndDate", 
                                             "designatedArea", 
                                             "designatedIncidentTypes"]
                                             ]

### Housing data

#### County-level data

Transform housing data
1. Join on FIPS code data to get FIPS code for each county. Extract county name and state abbreviation from "REGION" to join on FIPS code data.

In [11]:
# Strip trailing and leading whitespaces from all values
housing_county_df = housing_county_df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)

# Change "PERIOD_BEGIN" and "PERIOD_END" to datetime
housing_county_df["PERIOD_BEGIN"] = pd.to_datetime(housing_county_df["PERIOD_BEGIN"])
housing_county_df["PERIOD_END"] = pd.to_datetime(housing_county_df["PERIOD_END"])

# Track month of each observation
housing_county_df["MONTH"] = housing_county_df["PERIOD_BEGIN"].dt.to_period("M")

# Keep only the rows whose values pertain to all properties in the area, rather than to a specific property type
housing_county_df = housing_county_df[housing_county_df["PROPERTY_TYPE"] == "All Residential"]

# Helper functions to transform Redfin housing and FIPS code data into a format suitable for join
def clean_strict(text):
    """Standardizes names while PRESERVING 'city' and 'county' to distinguish between county names like Baltimore City/County."""
    if pd.isna(text): return ""
    text = str(text).lower().split(',')[0]
    # Redfin often uses 'City County' for independent cities; standardize to 'City'
    text = text.replace("city county", "city")
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return " ".join(text.split())

def clean_relaxed(text):
    """Standardizes names by REMOVING all geographic suffixes for a broader match."""
    if pd.isna(text): return ""
    text = str(text).lower().split(',')[0]
    suffixes = r'\b(county|parish|city|borough|municipality|census area|and|city county)\b'
    text = re.sub(suffixes, '', text)
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return " ".join(text.split())

# Join Redfin housing on FIPS code data
fips_df['clean_strict'] = fips_df['county_name'].apply(clean_strict)
fips_df['clean_relaxed'] = fips_df['county_name'].apply(clean_relaxed)
fips_df['clean_msa'] = fips_df['msa_name'].apply(clean_relaxed)
fips_df['clean_csa'] = fips_df['csa_name'].apply(clean_relaxed)

housing_county_df['clean_strict'] = housing_county_df['REGION'].apply(clean_strict)
housing_county_df['clean_relaxed'] = housing_county_df['REGION'].apply(clean_relaxed)
housing_county_df['clean_parent_metro'] = housing_county_df['PARENT_METRO_REGION'].apply(clean_relaxed)

fips_cols = ['fips', 'county_name', 'state', 'msa_name', 'csa_name']

# First, perform Strict County Match
stage1 = pd.merge(housing_county_df, fips_df[fips_cols + ['clean_strict']], 
                  left_on=['clean_strict', 'STATE_CODE'], right_on=['clean_strict', 'state'], how='left')

matched = stage1[stage1['fips'].notna()].copy()
unmatched = stage1[stage1['fips'].isna()].copy()
unmatched = unmatched.drop(columns=[c for c in fips_cols if c in unmatched.columns], errors='ignore')

# Next, perform Relaxed County Match (Handles missing suffixes) ---
stage2 = pd.merge(unmatched, fips_df[fips_cols + ['clean_relaxed']], 
                  left_on=['clean_relaxed', 'STATE_CODE'], right_on=['clean_relaxed', 'state'], how='left')

matched = pd.concat([matched, stage2[stage2['fips'].notna()]])
unmatched = stage2[stage2['fips'].isna()].copy().drop(columns=[c for c in fips_cols if c in stage2.columns], errors='ignore')

# Fallback 1: Join on Parent Metro -> MSA
stage3 = pd.merge(unmatched, fips_df[fips_cols + ['clean_msa']], 
                  left_on=['clean_parent_metro', 'STATE_CODE'], right_on=['clean_msa', 'state'], how='left')

matched = pd.concat([matched, stage3[stage3['fips'].notna()]])
unmatched = stage3[stage3['fips'].isna()].copy().drop(columns=[c for c in fips_cols if c in stage3.columns], errors='ignore')

# Fallback 2: Join on Parent Metro -> CSA
stage4 = pd.merge(unmatched, fips_df[fips_cols + ['clean_csa']], 
                  left_on=['clean_parent_metro', 'STATE_CODE'], right_on=['clean_csa', 'state'], how='left')

# Final Consolidation
housing_county_df = pd.concat([matched, stage4], ignore_index=True)
housing_county_df.drop(columns=['clean_strict', 'clean_relaxed', 'clean_parent_metro', 'state'], inplace=True, errors='ignore')

In [12]:
# Add FIPS codes manually for counties that couldn't be matched
housing_county_df.loc[housing_county_df['REGION'] == 'Maui County, HI', 'fips'] = '15009'
housing_county_df.loc[housing_county_df['REGION'] == 'La Salle Parish, LA', 'fips'] = '22059'

In [13]:
# Add county-level economic features to monthly housing observations.
# For housing months beyond the economic coverage window, use the latest available economic year as a proxy.
housing_county_df['fips'] = housing_county_df['fips'].astype(str).str.zfill(5)
housing_county_df['housing_year'] = housing_county_df['PERIOD_BEGIN'].dt.year.astype('Int64')

personal_income_county_df = (
    personal_income_df.assign(
        county_fips=personal_income_df['IBRC_GEO_ID'].astype(str).str.replace('.0', '', regex=False).str.zfill(5),
        Year=pd.to_numeric(personal_income_df['Year'], errors='coerce').astype('Int64'),
        Data=pd.to_numeric(personal_income_df['Data'], errors='coerce'),
        line_desc=personal_income_df['Linecode Description'].astype(str).str.strip(),
    )
    .loc[
        lambda df: df['county_fips'].str.len().eq(5)
        & (~df['county_fips'].eq('00000'))
        & (df['line_desc'] == 'Per capita personal income (dollars)'),
        ['county_fips', 'Year', 'Data'],
    ]
    .rename(columns={'Year': 'bea_year', 'Data': 'per_capita_income'})
    .drop_duplicates(subset=['county_fips', 'bea_year'])
)
latest_bea_year = int(personal_income_county_df['bea_year'].max())

employment_wages_county_df = read_cew_employment_wages_cached(
    ROOT / 'data' / 'economic' / 'CEW - US, States, Counties - Total Ownership.csv',
    cache_dir=CACHE_DIR,
)
latest_cew_year = int(employment_wages_county_df['cew_year'].max())

population_growth_county_df = (
    population_estimates_df[['fips', 'Year', 'Population']]
    .assign(
        population_year=lambda df: pd.to_numeric(df['Year'], errors='coerce').astype('Int64'),
        population=lambda df: pd.to_numeric(df['Population'], errors='coerce'),
    )
    .sort_values(['fips', 'population_year'])
    .drop_duplicates(subset=['fips', 'population_year'])
)
population_growth_county_df['population_growth_yoy'] = (
    population_growth_county_df.groupby('fips')['population'].pct_change()
)
population_growth_county_df = population_growth_county_df[[
    'fips',
    'population_year',
    'population',
    'population_growth_yoy',
]]
latest_population_year = int(population_growth_county_df['population_year'].max())

housing_county_df['bea_year'] = housing_county_df['housing_year'].clip(upper=latest_bea_year)
housing_county_df['cew_year'] = housing_county_df['housing_year'].clip(upper=latest_cew_year)
housing_county_df['population_year'] = housing_county_df['housing_year'].clip(upper=latest_population_year)

housing_county_df = (
    housing_county_df
    .merge(personal_income_county_df, left_on=['fips', 'bea_year'], right_on=['county_fips', 'bea_year'], how='left')
    .drop(columns=['county_fips'])
    .merge(employment_wages_county_df, left_on=['fips', 'cew_year'], right_on=['county_fips', 'cew_year'], how='left')
    .drop(columns=['county_fips'])
    .merge(population_growth_county_df, left_on=['fips', 'population_year'], right_on=['fips', 'population_year'], how='left')
)


In [14]:
# Select relevant columns after joining county housing to FIPS and economic data
housing_county_df = housing_county_df[["fips", 
                                       "REGION",
                                       "county_name",
                                       "STATE_CODE",
                                       "MONTH",
                                       "housing_year",
                                       "PERIOD_BEGIN",
                                       "PERIOD_END", 
                                       "per_capita_income",
                                       "employment",
                                       "average_wage_per_job",
                                       "population",
                                       "population_growth_yoy",
                                       "MEDIAN_PPSF", 
                                       "MEDIAN_PPSF_YOY", 
                                       "MEDIAN_LIST_PPSF", 
                                       "MEDIAN_LIST_PPSF_YOY", 
                                       "HOMES_SOLD", 
                                       "HOMES_SOLD_YOY",
                                       "PENDING_SALES",
                                       "PENDING_SALES_YOY",
                                       "INVENTORY",
                                       "INVENTORY_YOY",
                                       "MONTHS_OF_SUPPLY",
                                       "MONTHS_OF_SUPPLY_YOY",
                                       "MEDIAN_DOM",           # Days on Market
                                       "MEDIAN_DOM_YOY",
                                       "AVG_SALE_TO_LIST",
                                       "AVG_SALE_TO_LIST_YOY",
                                       "PRICE_DROPS",
                                       "PRICE_DROPS_YOY"
                                       ]
                                       ]

#### Measure change in velocities (YOY change) over months

The housing market index is an equal-weight composite of four year-over-year market metrics: Median PPSF, Avg Sale to List, Homes Sold, and Inventory. Each YOY component is standardized as a z-score across the county-month housing panel so metrics with different units contribute on the same scale. The index score is the row-wise mean of the available component z-scores. `HOUSING_MARKET_INDEX_MOM` is the county-level month-over-month change in that index score.

In [15]:
# Build the county housing analysis table and calculate month-over-month changes in YOY values
housing_df = housing_county_df.copy()
housing_df = housing_df.sort_values(["fips", "MONTH"])
HOUSING_MARKET_INDEX_COMPONENTS = [
    "MEDIAN_PPSF_YOY",
    "AVG_SALE_TO_LIST_YOY",
    "HOMES_SOLD_YOY",
    "INVENTORY_YOY",
]
housing_market_index_z_cols = []
for component_col in HOUSING_MARKET_INDEX_COMPONENTS:
    z_col = f"{component_col}_Z"
    component_values = pd.to_numeric(housing_df[component_col], errors = "coerce")
    component_std = component_values.std(skipna = True)
    housing_df[z_col] = (component_values - component_values.mean(skipna = True)) / component_std if pd.notna(component_std) and component_std else pd.NA
    housing_market_index_z_cols.append(z_col)

housing_df["HOUSING_MARKET_INDEX"] = housing_df[housing_market_index_z_cols].mean(axis = 1, skipna = True)
housing_df["HOUSING_MARKET_INDEX_MOM"] = housing_df.groupby("fips")["HOUSING_MARKET_INDEX"].diff()
housing_df = housing_df.drop(columns = housing_market_index_z_cols)
housing_df["MEDIAN_PPSF_YOY_MOM"] = housing_df.groupby("fips")["MEDIAN_PPSF_YOY"].diff()
housing_df["MEDIAN_LIST_PPSF_YOY_MOM"] = housing_df.groupby("fips")["MEDIAN_LIST_PPSF_YOY"].diff()
housing_df["HOMES_SOLD_YOY_MOM"] = housing_df.groupby("fips")["HOMES_SOLD_YOY"].diff()
housing_df["PENDING_SALES_YOY_MOM"] = housing_df.groupby("fips")["PENDING_SALES_YOY"].diff()
housing_df["INVENTORY_YOY_MOM"] = housing_df.groupby("fips")["INVENTORY_YOY"].diff()
housing_df["MONTHS_OF_SUPPLY_YOY_MOM"] = housing_df.groupby("fips")["MONTHS_OF_SUPPLY_YOY"].diff()
housing_df["MEDIAN_DOM_YOY_MOM"] = housing_df.groupby("fips")["MEDIAN_DOM_YOY"].diff()
housing_df["AVG_SALE_TO_LIST_YOY_MOM"] = housing_df.groupby("fips")["AVG_SALE_TO_LIST_YOY"].diff()


## County Socioeconomic Profiles

This section prepares the economic and demographic source data, then delegates county-profile clustering to `src/cluster_county_econ_demographics.py`. The notebook applies the returned county labels to `housing_df` for downstream visualization exports.

### Build and Apply County Profiles

The script builds the 10-year county panel, aggregates it to one averaged row per county, evaluates candidate models, writes profile artifacts, and returns county-level profile labels and interpretations.

In [16]:
from cluster_county_econ_demographics import (
    FEATURE_COLUMNS,
    build_county_profile_clusters,
)

county_profile_outputs = build_county_profile_clusters(
    bea_income_df=bea_income_df,
    cew_total_df=cew_total_df,
    population_change_df=population_change_df,
    population_age_sex_df=population_age_sex_df,
    population_race_df=population_race_df,
    population_estimates_df=population_estimates_df,
    target_years=TARGET_YEARS,
    output_dir=ROOT / 'output' / 'visualizations',
)

county_year_panel = county_profile_outputs['county_year_panel']
county_average_df = county_profile_outputs['county_average_df']
cluster_input = county_profile_outputs['cluster_input']
cluster_quality_df = county_profile_outputs['cluster_quality_df']
county_profiles_summary_df = county_profile_outputs['county_profiles_summary_df']
county_profile_df = county_profile_outputs['county_profile_df']
cluster_interpretations_df = county_profile_outputs['cluster_interpretations_df']
best_cluster_row = county_profile_outputs['best_cluster_row']
best_algorithm = county_profile_outputs['best_algorithm']
best_k = county_profile_outputs['best_k']

housing_df = (
    housing_df.drop(columns=[
        'county_profile',
        'county_profile_desc',
        'county_profile_algorithm',
        'county_profile_k',
        'county_profile_silhouette',
        'county_profile_calinski_harabasz',
        'county_profile_davies_bouldin',
        'county_profile_combined_metric_rank',
    ], errors='ignore')
    .merge(
        county_profile_df.drop(columns='county_name'),
        on='fips',
        how='left',
    )
)

globals().pop('_housing_lookup_cache', None)
globals().pop('_prepared_incident_df_cache', None)

county_profile_outputs['paths']

{'profiles': WindowsPath('c:/Users/kenny/Data Science Projects/quoll-intelligence/output/visualizations/county_profiles.csv'),
 'assignments': WindowsPath('c:/Users/kenny/Data Science Projects/quoll-intelligence/output/visualizations/county_profile_assignments.csv'),
 'model': WindowsPath('c:/Users/kenny/Data Science Projects/quoll-intelligence/output/visualizations/county_profile_model.joblib'),
 'quality': WindowsPath('c:/Users/kenny/Data Science Projects/quoll-intelligence/output/visualizations/county_profile_model_quality.csv')}

In [17]:
panel_diagnostics = pd.DataFrame({
    'county_year_rows': [len(county_year_panel)],
    'distinct_counties': [county_year_panel['fips'].nunique()],
    'analysis_years': [f"{county_year_panel['Year'].min()}-{county_year_panel['Year'].max()}"],
    'counties_with_bea': [county_year_panel.loc[county_year_panel['personal_income'].notna(), 'fips'].nunique()],
    'counties_with_cew': [county_year_panel.loc[county_year_panel['Employment'].notna(), 'fips'].nunique()],
})
panel_diagnostics

,county_year_rows,distinct_counties,analysis_years,counties_with_bea,counties_with_cew
0,31580,3158,2016-2025,566,576


### County-Level Average Dataset

Inspect the county-level feature table returned by the clustering script.

In [18]:
county_average_df.head()

,fips,county_name,log_population,per_capita_income,employment_rate_proxy,Average Weekly Wage,transfer_receipts_share,dividends_rent_share,proprietors_income_share,youth_share,prime_working_age_share,senior_share,male_share,white_share,black_share,asian_share,hispanic_share,natural_increase_rate,international_migration_rate,domestic_migration_rate
0,01001,"Autauga County, AL",10.9737,"46,585.2000",0.1948,841.6000,0.2296,0.1464,0.0387,0.2357,0.5262,0.1600,0.4862,0.7557,0.2063,0.0124,0.0344,0.0014,0.0006,0.0052
1,01003,"Baldwin County, AL",12.3696,"52,898.7000",0.3285,823.8000,0.2221,0.2188,0.0596,0.2130,0.5042,0.2115,0.4864,0.8731,0.0872,0.0112,0.0531,-0.0007,0.0015,0.0246
2,01005,"Barbour County, AL",10.1218,"37,451.9000",0.3171,779.3000,0.3692,0.1491,0.0658,0.2086,0.5112,0.1999,0.5274,0.4954,0.4759,0.0050,0.0551,-0.0024,0.0013,-0.0071
3,01007,"Bibb County, AL",10.0111,"34,948.0000",0.2057,884.4000,0.3274,0.1002,0.0334,0.2028,0.5472,0.1718,0.5321,0.7653,0.2118,0.0026,0.0313,-0.0014,0.0005,-0.0007
4,01009,"Blount County, AL",10.9828,"39,800.3000",0.1467,761.6000,0.2649,0.1191,0.0664,0.2305,0.5056,0.1864,0.4957,0.9557,0.0183,0.0038,0.0991,-0.0013,0.0003,0.0033


In [19]:
county_average_diagnostics = pd.DataFrame({
    'county_rows': [len(county_average_df)],
    'distinct_counties': [county_average_df['fips'].nunique()],
    'avg_missing_features_per_county': [county_average_df[FEATURE_COLUMNS].isna().sum(axis=1).mean()],
})
county_average_diagnostics

,county_rows,distinct_counties,avg_missing_features_per_county
0,3158,3158,4.9227


### Cluster Input Diagnostics

Inspect feature coverage and the number of counties that entered the clustering script.

In [20]:
cluster_input[FEATURE_COLUMNS].isna().sum().sort_values(ascending=False).head(12)

transfer_receipts_share     2594
dividends_rent_share        2594
proprietors_income_share    2594
per_capita_income           2592
employment_rate_proxy       2582
Average Weekly Wage         2582
male_share                     2
youth_share                    2
prime_working_age_share        2
senior_share                   2
log_population                 0
white_share                    0
dtype: int64

In [21]:
cluster_input.shape

(3158, 21)

#### Model Selection Output

The model-quality table is generated by `cluster_county_econ_demographics.py`. The notebook only displays the selected-model diagnostics returned by the script.

In [22]:
cluster_quality_df

,algorithm,k,silhouette_score,calinski_harabasz_score,davies_bouldin_index,cluster_sizes,silhouette_rank,calinski_harabasz_rank,davies_bouldin_rank,combined_metric_rank
0,kmeans,9,0.9051,"11,872.6351",0.3452,"{0: 2733, 1: 20, 2: 73, 3: 6, 4: 191, 5: 2, 6:...",4.0000,1.0000,2.0000,2.3333
1,kmeans,8,0.9039,"10,680.3728",0.3357,"{0: 2784, 1: 52, 2: 1, 3: 6, 4: 101, 5: 192, 6...",6.0000,2.0000,1.0000,3.0000
2,kmeans,7,0.9039,"9,656.8304",0.3718,"{0: 101, 1: 2782, 2: 22, 3: 194, 4: 52, 5: 5, ...",5.0000,4.0000,3.0000,4.0000
3,kmeans,4,0.9067,"5,791.5740",0.4240,"{0: 3002, 1: 120, 2: 7, 3: 29}",2.0000,11.0000,7.0000,6.6667
4,kmeans,6,0.9020,"9,199.0830",0.4278,"{0: 2788, 1: 52, 2: 5, 3: 101, 4: 22, 5: 190}",7.0000,5.0000,8.0000,6.6667
5,ward_agglomerative,7,0.8913,"8,850.9545",0.4052,"{0: 243, 1: 22, 2: 5, 3: 2745, 4: 60, 5: 81, 6...",9.0000,7.0000,5.0000,7.0000
6,ward_agglomerative,9,0.8853,"9,818.4192",0.3853,"{0: 22, 1: 2745, 2: 99, 3: 144, 4: 60, 5: 81, ...",14.0000,3.0000,4.0000,7.0000
7,kmeans,5,0.8959,"7,957.0534",0.4185,"{0: 2812, 1: 29, 2: 120, 3: 7, 4: 190}",8.0000,9.0000,6.0000,7.6667
8,kmeans,3,0.9099,"5,671.1544",0.5028,"{0: 3007, 1: 27, 2: 124}",1.0000,12.0000,13.0000,8.6667
9,ward_agglomerative,6,0.8886,"8,457.5579",0.4657,"{0: 245, 1: 22, 2: 5, 3: 2745, 4: 60, 5: 81}",10.0000,8.0000,11.0000,9.6667


In [23]:
plot_quality_df = cluster_quality_df.melt(
    id_vars=['algorithm', 'k'],
    value_vars=['silhouette_score', 'calinski_harabasz_score', 'davies_bouldin_index'],
    var_name='metric',
    value_name='score',
)
plot_quality_df.head()

,algorithm,k,metric,score
0,kmeans,9,silhouette_score,0.9051
1,kmeans,8,silhouette_score,0.9039
2,kmeans,7,silhouette_score,0.9039
3,kmeans,4,silhouette_score,0.9067
4,kmeans,6,silhouette_score,0.9020


#### Assigned County Profiles

In [24]:
county_profile_df['county_profile'].value_counts().sort_index().to_frame('counties')

,counties
county_profile,
0,2733
1,20
2,73
3,6
4,191
5,2
6,47
7,85
8,1


Preview assigned county profile labels returned by the script.

In [25]:
county_profile_df.head()

,fips,county_name,county_profile,county_profile_desc,county_profile_algorithm,county_profile_k,county_profile_silhouette,county_profile_calinski_harabasz,county_profile_davies_bouldin,county_profile_combined_metric_rank
0,01001,"Autauga County, AL",0,"This cluster contains 2,733 counties. It is di...",kmeans,9,0.9051,"11,872.6351",0.3452,2.3333
1,01003,"Baldwin County, AL",7,This cluster contains 85 counties. It is disti...,kmeans,9,0.9051,"11,872.6351",0.3452,2.3333
2,01005,"Barbour County, AL",4,This cluster contains 191 counties. It is dist...,kmeans,9,0.9051,"11,872.6351",0.3452,2.3333
3,01007,"Bibb County, AL",4,This cluster contains 191 counties. It is dist...,kmeans,9,0.9051,"11,872.6351",0.3452,2.3333
4,01009,"Blount County, AL",4,This cluster contains 191 counties. It is dist...,kmeans,9,0.9051,"11,872.6351",0.3452,2.3333


#### County Profile Summaries

In [26]:
county_profiles_summary_df

,county_profile,county_profile_desc,county_profile_algorithm,county_profile_k,county_profile_silhouette,county_profile_calinski_harabasz,county_profile_davies_bouldin,county_profile_combined_metric_rank
0,0,"This cluster contains 2,733 counties. It is di...",kmeans,9,0.9051,"11,872.6351",0.3452,2.3333
1,1,This cluster contains 20 counties. It is disti...,kmeans,9,0.9051,"11,872.6351",0.3452,2.3333
2,2,This cluster contains 73 counties. It is disti...,kmeans,9,0.9051,"11,872.6351",0.3452,2.3333
3,3,This cluster contains 6 counties. It is distin...,kmeans,9,0.9051,"11,872.6351",0.3452,2.3333
4,4,This cluster contains 191 counties. It is dist...,kmeans,9,0.9051,"11,872.6351",0.3452,2.3333
5,5,This cluster contains 2 counties. It is distin...,kmeans,9,0.9051,"11,872.6351",0.3452,2.3333
6,6,This cluster contains 47 counties. It is disti...,kmeans,9,0.9051,"11,872.6351",0.3452,2.3333
7,7,This cluster contains 85 counties. It is disti...,kmeans,9,0.9051,"11,872.6351",0.3452,2.3333
8,8,This cluster contains 1 counties. It is distin...,kmeans,9,0.9051,"11,872.6351",0.3452,2.3333


#### Plain-Language County Profile Interpretations

In [27]:
layman_profile_interpretations_df = pd.DataFrame([
    {
        "county_profile": 0,
        "plain_language_profile": 'Mainstream counties with modest incomes',
        "layman_interpretation": 'Most counties fall here. These places look close to the national county average, with somewhat more reliance on transfer income and somewhat lower incomes, investment income, and wages.',
        "example_counties": 'Autauga County, AL; Butler County, AL; Choctaw County, AL',
    },
    {
        "county_profile": 1,
        "plain_language_profile": 'High-income job centers',
        "layman_interpretation": 'Small group of prosperous counties with high personal income, strong wages, and more income from investments. They tend to rely less on transfer payments.',
        "example_counties": 'Alameda County, CA; Denver County, CO; Eagle County, CO',
    },
    {
        "county_profile": 2,
        "plain_language_profile": 'Large diverse metros',
        "layman_interpretation": 'Metropolitan and regional hub counties with higher incomes, more working-age residents, and more racial and ethnic diversity than the typical county.',
        "example_counties": 'Jefferson County, AL; Madison County, AL; Shelby County, AL',
    },
    {
        "county_profile": 3,
        "plain_language_profile": 'Ultra-high-income tech and finance counties',
        "layman_interpretation": 'A very small set of wealthy, high-wage counties anchored by major technology, finance, or expensive coastal economies.',
        "example_counties": 'Marin County, CA; San Francisco County, CA; Santa Clara County, CA',
    },
    {
        "county_profile": 4,
        "plain_language_profile": 'Lower-income rural counties',
        "layman_interpretation": 'Mostly smaller rural counties with lower incomes, lower wages, older populations, and greater reliance on transfer payments.',
        "example_counties": 'Barbour County, AL; Bibb County, AL; Blount County, AL',
    },
    {
        "county_profile": 5,
        "plain_language_profile": 'Remote fast-growth Alaska counties',
        "layman_interpretation": 'Two remote Alaska boroughs with unusual population-change patterns, including high natural increase and international migration but low domestic migration and income.',
        "example_counties": 'Skagway-Hoonah-Angoon Borough, AK; Wrangell-Petersburg Borough, AK',
    },
    {
        "county_profile": 6,
        "plain_language_profile": 'Affluent Alaska and coastal counties',
        "layman_interpretation": 'Higher-income counties with more working-age residents and less reliance on transfer payments, including several Alaska boroughs and coastal places.',
        "example_counties": 'Anchorage Borough, AK; Juneau Borough, AK; Kodiak Island Borough, AK',
    },
    {
        "county_profile": 7,
        "plain_language_profile": 'Growing recreation and Sun Belt counties',
        "layman_interpretation": 'Counties with modestly stronger employment, more youth, and more Hispanic residents. Many are fast-growing recreation, retirement, or Sun Belt counties.',
        "example_counties": 'Baldwin County, AL; Coconino County, AZ; Pima County, AZ',
    },
    {
        "county_profile": 8,
        "plain_language_profile": 'Resort wealth outlier',
        "layman_interpretation": 'A single resort-oriented county with extremely high incomes and investment income, a large working-age share, and relatively few children.',
        "example_counties": 'Pitkin County, CO',
    },
])

layman_profile_interpretations_df


,county_profile,plain_language_profile,layman_interpretation,example_counties
0,0,Mainstream counties with modest incomes,Most counties fall here. These places look clo...,"Autauga County, AL; Butler County, AL; Choctaw..."
1,1,High-income job centers,Small group of prosperous counties with high p...,"Alameda County, CA; Denver County, CO; Eagle C..."
2,2,Large diverse metros,Metropolitan and regional hub counties with hi...,"Jefferson County, AL; Madison County, AL; Shel..."
3,3,Ultra-high-income tech and finance counties,"A very small set of wealthy, high-wage countie...","Marin County, CA; San Francisco County, CA; Sa..."
4,4,Lower-income rural counties,Mostly smaller rural counties with lower incom...,"Barbour County, AL; Bibb County, AL; Blount Co..."
5,5,Remote fast-growth Alaska counties,Two remote Alaska boroughs with unusual popula...,"Skagway-Hoonah-Angoon Borough, AK; Wrangell-Pe..."
6,6,Affluent Alaska and coastal counties,Higher-income counties with more working-age r...,"Anchorage Borough, AK; Juneau Borough, AK; Kod..."
7,7,Growing recreation and Sun Belt counties,"Counties with modestly stronger employment, mo...","Baldwin County, AL; Coconino County, AZ; Pima ..."
8,8,Resort wealth outlier,A single resort-oriented county with extremely...,"Pitkin County, CO"


In [28]:
for row in layman_profile_interpretations_df.itertuples(index=False):
    print(f"Profile {row.county_profile}: {row.plain_language_profile}")
    print(row.layman_interpretation)
    print(f"Example counties: {row.example_counties}")
    print('-' * 100)


Profile 0: Mainstream counties with modest incomes
Most counties fall here. These places look close to the national county average, with somewhat more reliance on transfer income and somewhat lower incomes, investment income, and wages.
Example counties: Autauga County, AL; Butler County, AL; Choctaw County, AL
----------------------------------------------------------------------------------------------------
Profile 1: High-income job centers
Small group of prosperous counties with high personal income, strong wages, and more income from investments. They tend to rely less on transfer payments.
Example counties: Alameda County, CA; Denver County, CO; Eagle County, CO
----------------------------------------------------------------------------------------------------
Profile 2: Large diverse metros
Metropolitan and regional hub counties with higher incomes, more working-age residents, and more racial and ethnic diversity than the typical county.
Example counties: Jefferson County, AL;

#### Written Artifacts

The script writes county profile summaries, county assignments, model quality metrics, and the selected model object under `output/visualizations`.

In [29]:
print(county_profile_outputs['paths']['profiles'])
print(county_profile_outputs['paths']['assignments'])
print(county_profile_outputs['paths']['model'])
county_profile_df.head()

c:\Users\kenny\Data Science Projects\quoll-intelligence\output\visualizations\county_profiles.csv
c:\Users\kenny\Data Science Projects\quoll-intelligence\output\visualizations\county_profile_assignments.csv
c:\Users\kenny\Data Science Projects\quoll-intelligence\output\visualizations\county_profile_model.joblib


,fips,county_name,county_profile,county_profile_desc,county_profile_algorithm,county_profile_k,county_profile_silhouette,county_profile_calinski_harabasz,county_profile_davies_bouldin,county_profile_combined_metric_rank
0,01001,"Autauga County, AL",0,"This cluster contains 2,733 counties. It is di...",kmeans,9,0.9051,"11,872.6351",0.3452,2.3333
1,01003,"Baldwin County, AL",7,This cluster contains 85 counties. It is disti...,kmeans,9,0.9051,"11,872.6351",0.3452,2.3333
2,01005,"Barbour County, AL",4,This cluster contains 191 counties. It is dist...,kmeans,9,0.9051,"11,872.6351",0.3452,2.3333
3,01007,"Bibb County, AL",4,This cluster contains 191 counties. It is dist...,kmeans,9,0.9051,"11,872.6351",0.3452,2.3333
4,01009,"Blount County, AL",4,This cluster contains 191 counties. It is dist...,kmeans,9,0.9051,"11,872.6351",0.3452,2.3333


## Cluster County Housing Market YOY Responses by Incident Type

This section delegates incident-type housing response clustering to `src/cluster_housing_yoy_responses.py`. The notebook applies the returned response cluster labels and interpretations during visualization export.

### Housing Response Cluster Artifacts

In [30]:
from cluster_housing_yoy_responses import (
    PPSF_RESPONSE_FEATURES,
    PPSF_RESPONSE_METRIC_LABELS,
    PPSF_RESPONSE_METRICS,
    build_all_housing_market_response_clusters,
)

housing_response_cluster_outputs = build_all_housing_market_response_clusters(
    natural_disasters_df=natural_disasters_df,
    housing_df=housing_df,
    output_dir=ROOT / 'output' / 'visualizations',
)

ppsf_response_cluster_results = housing_response_cluster_outputs['ppsf_response_cluster_results']
ppsf_response_cluster_comparison_df = housing_response_cluster_outputs['ppsf_response_cluster_comparison_df']
ppsf_response_cluster_summary_df = housing_response_cluster_outputs['ppsf_response_cluster_summary_df']
ppsf_response_best_cluster_rows = housing_response_cluster_outputs['ppsf_response_best_cluster_rows']
ppsf_response_cluster_annotations_df = housing_response_cluster_outputs['ppsf_response_cluster_annotations_df']
ppsf_response_cluster_interpretations_df = housing_response_cluster_outputs['ppsf_response_cluster_interpretations_df']

housing_response_cluster_outputs['paths']

{'assignments': WindowsPath('c:/Users/kenny/Data Science Projects/quoll-intelligence/output/visualizations/ppsf_response_cluster_assignments.csv'),
 'interpretations': WindowsPath('c:/Users/kenny/Data Science Projects/quoll-intelligence/output/visualizations/ppsf_response_cluster_interpretations.csv'),
 'comparison': WindowsPath('c:/Users/kenny/Data Science Projects/quoll-intelligence/output/visualizations/ppsf_response_cluster_comparison.csv'),
 'summary': WindowsPath('c:/Users/kenny/Data Science Projects/quoll-intelligence/output/visualizations/ppsf_response_cluster_summary.csv')}

### Selected Housing Response Models

For each incident type, the script evaluates Ward agglomerative and KMeans candidates, selects the best model by silhouette score, and returns the final county labels and cluster interpretations.

In [31]:
ppsf_response_best_cluster_rows

,incident_type,algorithm,k,status,counties_clustered,complete_county_incident_vectors,feature_count,metrics_used,silhouette_score,balance_penalty,balanced_selection_score,min_cluster_size,tiny_cluster_count,smallest_cluster_share,largest_cluster_share,cluster_size_cv,required_min_cluster_size,has_tiny_cluster,cluster_col,cluster_sizes,is_selectable,effective_selection_score
0,Coastal Storm,ward_agglomerative,2,clustered,35,88,28,"Median PPSF, Avg Sale to List, Homes Sold, Inv...",0.5298,1.1643,0.1223,1,1,0.0286,0.9714,0.9429,3,True,median_ppsf_response_cluster_k2,"{0: 34, 1: 1}",False,-inf
11,Earthquake,kmeans_benchmark,3,clustered,62,69,28,"Median PPSF, Avg Sale to List, Homes Sold, Inv...",0.5179,1.4139,0.0230,1,1,0.0161,0.9194,1.2446,4,True,median_ppsf_response_kmeans_cluster_k3,"{0: 4, 1: 57, 2: 1}",False,-inf
17,Fire,kmeans_benchmark,2,clustered,270,1338,28,"Median PPSF, Avg Sale to List, Homes Sold, Inv...",0.4405,0.6944,0.1974,50,0,0.1852,0.8148,0.6296,14,False,median_ppsf_response_kmeans_cluster_k2,"{0: 50, 1: 220}",True,0.1974
26,Flood,ward_agglomerative,3,clustered,937,3549,28,"Median PPSF, Avg Sale to List, Homes Sold, Inv...",0.3630,0.9299,0.0376,1,1,0.0011,0.7471,0.9299,47,True,median_ppsf_response_cluster_k3,"{0: 236, 1: 1, 2: 700}",False,-inf
34,Hurricane,ward_agglomerative,3,clustered,790,5342,28,"Median PPSF, Avg Sale to List, Homes Sold, Inv...",0.4288,1.0273,0.0693,1,1,0.0013,0.7848,0.9925,40,True,median_ppsf_response_cluster_k3,"{0: 169, 1: 1, 2: 620}",False,-inf
40,Mud/Landslide,ward_agglomerative,2,clustered,74,145,28,"Median PPSF, Avg Sale to List, Homes Sold, Inv...",0.5660,0.9257,0.2420,8,0,0.1081,0.8919,0.7838,4,False,median_ppsf_response_cluster_k2,"{0: 8, 1: 66}",True,0.2420
49,Severe Ice Storm,kmeans_benchmark,2,clustered,400,652,28,"Median PPSF, Avg Sale to List, Homes Sold, Inv...",0.3911,0.6950,0.1478,74,0,0.1850,0.8150,0.6300,20,False,median_ppsf_response_kmeans_cluster_k2,"{0: 326, 1: 74}",True,0.1478
57,Severe Storm,kmeans_benchmark,2,clustered,1162,2965,28,"Median PPSF, Avg Sale to List, Homes Sold, Inv...",0.4232,0.6975,0.1791,214,0,0.1842,0.8158,0.6317,59,False,median_ppsf_response_kmeans_cluster_k2,"{0: 948, 1: 214}",True,0.1791
65,Snowstorm,kmeans_benchmark,2,clustered,143,156,28,"Median PPSF, Avg Sale to List, Homes Sold, Inv...",0.5785,1.0192,0.2217,11,0,0.0769,0.9231,0.8462,8,False,median_ppsf_response_kmeans_cluster_k2,"{0: 132, 1: 11}",True,0.2217
73,Tornado,kmeans_benchmark,2,clustered,119,152,28,"Median PPSF, Avg Sale to List, Homes Sold, Inv...",0.4353,0.8214,0.1478,17,0,0.1429,0.8571,0.7143,6,False,median_ppsf_response_kmeans_cluster_k2,"{0: 17, 1: 102}",True,0.1478


## Build Incident-Housing Datasets

Create subset DataFrames of county housing data that fall within the incident window for each incident type, excluding `Biological`, `Chemical`, `Other`, `Human Cause`, `Terrorist`, `Fishing Losses`, `Dam/Levee Break`, and `Toxic Substances`.
State-level incidents are mapped to county housing rows through the county lookup keyed by county FIPS and state prefix.

Transformation steps:
- Impute missing `incidentEndDate` values with the estimated end date based on the median duration of that incident type
- Get the monthly periods for natural disasters to determine the incident window to filter on housing data
- Measure the monthly change in YOY values


In [32]:
'''
Get subsets of housing data that falls within incident window using pandas dataframe
get_housing_data_in_window(): Returns county-level housing data for the period of 12-months before incident start to 12-months after incident end
get_housing_data_in_window2(): Returns county-level housing data for the period of 12-months before incident start to 24-months after incident end
'''

ECONOMIC_BIN_COUNT = 5

def _add_quantile_bins(df, source_col, target_col, bins = ECONOMIC_BIN_COUNT):
    numeric_values = pd.to_numeric(df[source_col], errors = "coerce")
    result = pd.Series(pd.NA, index = df.index, dtype = "Int64")
    valid_mask = numeric_values.notna()
    if valid_mask.sum() == 0:
        df[target_col] = result
        return df

    try:
        binned = pd.qcut(numeric_values[valid_mask], q = bins, labels = False, duplicates = "drop")
    except ValueError:
        df[target_col] = result
        return df

    result.loc[valid_mask] = pd.Series(binned, index = numeric_values[valid_mask].index).astype("Int64") + 1
    df[target_col] = result
    return df

def _get_housing_lookup_cache():
    cache = globals().get("_housing_lookup_cache")
    if cache is not None:
        return cache

    housing_with_keys = housing_df.copy()
    housing_with_keys["fips_normalized"] = housing_with_keys["fips"].astype(str).str.zfill(5)
    housing_with_keys["state_prefix"] = housing_with_keys["fips_normalized"].str[:2]
    housing_with_keys["is_county"] = ~housing_with_keys["fips_normalized"].str.endswith("000")
    housing_with_keys = _add_quantile_bins(housing_with_keys, "per_capita_income", "per_capita_income_bin")
    housing_with_keys = _add_quantile_bins(housing_with_keys, "employment", "employment_bin")
    housing_with_keys = _add_quantile_bins(housing_with_keys, "average_wage_per_job", "average_wage_per_job_bin")

    county_rows = housing_with_keys[housing_with_keys["is_county"]].copy()
    cache = {
        "county_rows": county_rows,
        "county_months": county_rows.groupby("fips_normalized", sort = False)["MONTH"].agg(lambda values: set(values)),
        "state_months": county_rows.groupby("state_prefix", sort = False)["MONTH"].agg(lambda values: set(values)),
        "empty": housing_with_keys.iloc[0:0].copy(),
    }
    globals()["_housing_lookup_cache"] = cache
    return cache

EXCLUDED_INCIDENT_TYPES = {
    "Biological",
    "Chemical",
    "Other",
    "Human Cause",
    "Terrorist",
    "Fishing Losses",
    "Dam/Levee Break",
    "Toxic Substances",
}

def _prepare_incident_df(incident_type = None):
    cache = globals().setdefault("_prepared_incident_df_cache", {})
    cache_key = incident_type or "__all__"
    if cache_key in cache:
        return cache[cache_key]

    if incident_type in EXCLUDED_INCIDENT_TYPES:
        raise ValueError("incident_type is not valid.")

    if incident_type is not None and (natural_disasters_df["incidentType"] == incident_type).any() is False:
        raise ValueError("incident_type is not valid.")

    cutoff_date = pd.Timestamp(year = pd.Timestamp.now().year - 10, month = 1, day = 1)
    incident_df = natural_disasters_df.loc[
        natural_disasters_df["incidentBeginDate"] >= cutoff_date
    ].copy()
    incident_df = incident_df.loc[~incident_df["incidentType"].isin(EXCLUDED_INCIDENT_TYPES)].copy()
    if incident_type is not None:
        incident_df = incident_df.loc[incident_df["incidentType"] == incident_type].copy()

    incident_duration = incident_df["incidentEndDate"] - incident_df["incidentBeginDate"]
    incident_median_duration = incident_duration[incident_duration.notna()].median()
    incident_df["incidentEndDate"] = incident_df["incidentEndDate"].fillna(
        incident_df["incidentBeginDate"] + incident_median_duration
    )
    incident_df["incident_begin_month"] = incident_df["incidentBeginDate"].dt.to_period("M")
    incident_df["incident_end_month"] = incident_df["incidentEndDate"].dt.to_period("M")

    cache[cache_key] = incident_df
    return incident_df

def _build_incident_housing_subset(incident_df, months_before, months_after, after_anchor = "end", complete_after_anchor = None):
    housing_lookup = _get_housing_lookup_cache()
    county_rows = housing_lookup["county_rows"]
    county_months = housing_lookup["county_months"]
    state_months = housing_lookup["state_months"]
    empty_result = housing_lookup["empty"].copy()

    event_month_rows = []
    for incident_num, event in enumerate(incident_df.itertuples(index = False), start = 1):
        fips = str(event.fips).zfill(5)
        event_start = event.incident_begin_month
        event_end = event.incident_end_month

        if fips.endswith("000"):
            available_months = state_months.get(fips[:2])
        else:
            available_months = county_months.get(fips)
        if available_months is None:
            continue

        if complete_after_anchor is not None:
            completion_anchor = event_end if complete_after_anchor == "end" else event_start
            required_after_months = {completion_anchor + offset for offset in range(1, months_after + 1)}
            if not required_after_months.issubset(available_months):
                continue

        after_reference = event_end if after_anchor == "end" else event_start
        offset_lookup = {
            event_start + offset: offset
            for offset in range(-months_before, 0)
        }
        offset_lookup[event_end] = 0
        offset_lookup.update({
            after_reference + offset: offset
            for offset in range(1, months_after + 1)
        })

        for month, offset in offset_lookup.items():
            event_month_rows.append({
                "event_fips": fips,
                "event_state_prefix": fips[:2],
                "is_statewide_event": fips.endswith("000"),
                "MONTH": month,
                "month_offset_from_incident": offset,
                "incident_num": incident_num,
                "incident_type": event.incidentType,
            })

    if not event_month_rows:
        incident_housing_df = empty_result
        incident_housing_df["month_offset_from_incident"] = pd.Series(dtype = "Int64")
        incident_housing_df["incident_num"] = pd.Series(dtype = "Int64")
        incident_housing_df["incident_type"] = pd.Series(dtype = "object")
    else:
        event_months = pd.DataFrame(event_month_rows)
        matched_frames = []
        county_events = event_months.loc[~event_months["is_statewide_event"]]
        state_events = event_months.loc[event_months["is_statewide_event"]]
        if not county_events.empty:
            matched_frames.append(
                county_rows.merge(
                    county_events,
                    left_on = ["fips_normalized", "MONTH"],
                    right_on = ["event_fips", "MONTH"],
                    how = "inner",
                )
            )
        if not state_events.empty:
            matched_frames.append(
                county_rows.merge(
                    state_events,
                    left_on = ["state_prefix", "MONTH"],
                    right_on = ["event_state_prefix", "MONTH"],
                    how = "inner",
                )
            )
        if matched_frames:
            incident_housing_df = pd.concat(matched_frames, ignore_index = True)
        else:
            incident_housing_df = empty_result
            incident_housing_df["month_offset_from_incident"] = pd.Series(dtype = "Int64")
            incident_housing_df["incident_num"] = pd.Series(dtype = "Int64")
            incident_housing_df["incident_type"] = pd.Series(dtype = "object")

    incident_housing_df = incident_housing_df.drop(
        columns = ["fips_normalized", "state_prefix", "is_county", "event_fips", "event_state_prefix", "is_statewide_event"],
        errors = "ignore",
    )
    incident_housing_df = incident_housing_df.merge(nri_county_df, on = "fips", how = "left")
    return incident_housing_df

def get_housing_data_in_window(incident_type = None, months = 12):
    try:
        incident_df = _prepare_incident_df(incident_type = incident_type)
    except ValueError:
        return f'incident_type is not valid.'

    return _build_incident_housing_subset(
        incident_df = incident_df,
        months_before = months,
        months_after = months,
        after_anchor = "end",
    )

def get_housing_data_in_window2(incident_type = None, months_before = 12, months_after = 24):
    if months_before < 1 or months_after < 1:
        raise ValueError("months_before and months_after must both be positive integers.")

    incident_df = _prepare_incident_df(incident_type = incident_type)
    return _build_incident_housing_subset(
        incident_df = incident_df,
        months_before = months_before,
        months_after = months_after,
        after_anchor = "end",
        complete_after_anchor = "end",
    )

## Build and Serve Visualization Web Pages

In [33]:

'''
Save incident subsets beside the web visualization and serve the maps locally
'''

from functools import partial
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler
import json
from pathlib import Path
import re
import socket
import threading

YOY_12_TO_24_METRICS = {
    "HOUSING_MARKET_INDEX": "HOUSING_MARKET_INDEX_MOM",
    "AVG_SALE_TO_LIST": "AVG_SALE_TO_LIST_YOY_MOM",
    "HOMES_SOLD": "HOMES_SOLD_YOY_MOM",
    "INVENTORY": "INVENTORY_YOY_MOM",
    "MEDIAN_PPSF": "MEDIAN_PPSF_YOY_MOM",
}
YOY_12_TO_24_DROP_COLUMNS = [
    "employment",
    "average_wage_per_job",
    "employment_bin",
    "average_wage_per_job_bin",
    "nri_risk_score",
    "nri_risk_rating",
    "nri_risk_rating_date",
]
YOY_12_TO_24_BIN_LABELS = {
    0: "significant_decrease",
    1: "moderate_to_no_change",
    2: "significant_increase",
}
REQUESTED_HOUSING_METRICS = [
    "AVG_SALE_TO_LIST",
    "HOMES_SOLD",
    "INVENTORY",
    "MEDIAN_DOM",
    "MEDIAN_LIST_PPSF",
    "MEDIAN_PPSF",
    "MONTHS_OF_SUPPLY",
    "PENDING_SALES",
    "PRICE_DROPS",
]
INDEX_HOUSING_COLUMNS = [
    "fips",
    "REGION",
    "county_name",
    "STATE_CODE",
    "MONTH",
    "housing_year",
    "PERIOD_BEGIN",
    "PERIOD_END",
    "per_capita_income",
    "employment",
    "average_wage_per_job",
    *REQUESTED_HOUSING_METRICS,
    "MEDIAN_PPSF_YOY",
    "AVG_SALE_TO_LIST_YOY",
    "HOMES_SOLD_YOY",
    "INVENTORY_YOY",
    "HOUSING_MARKET_INDEX",
    "HOUSING_MARKET_INDEX_MOM",
    "month_offset_from_incident",
    "incident_num",
    "incident_type",
    "county_profile",
    "nri_risk_rating",
]
INDEX_24M_COLUMNS = [
    "fips",
    "REGION",
    "county_name",
    "STATE_CODE",
    "MONTH",
    "housing_year",
    "PERIOD_BEGIN",
    "PERIOD_END",
    "per_capita_income",
    *REQUESTED_HOUSING_METRICS,
    "MEDIAN_PPSF_YOY",
    "AVG_SALE_TO_LIST_YOY",
    "HOMES_SOLD_YOY",
    "INVENTORY_YOY",
    "HOUSING_MARKET_INDEX",
    *[f"{metric}_YOY_MOM" for metric in REQUESTED_HOUSING_METRICS],
    "HOUSING_MARKET_INDEX_MOM",
    "month_offset_from_incident",
    "incident_num",
    "incident_type",
    "county_profile",
]
STORY_1_COLUMNS = [
    "fips",
    "REGION",
    "county_name",
    "MONTH",
    "PERIOD_BEGIN",
    "per_capita_income",
    "population_growth_yoy",
    "HOUSING_MARKET_INDEX",
    "HOUSING_MARKET_INDEX_MOM",
    "month_offset_from_incident",
    "incident_num",
    "county_profile",
    "nri_risk_rating",
    "nri_risk_rating_date",
]
STORY_2_COLUMNS = [
    "fips",
    "REGION",
    "county_name",
    "MONTH",
    "housing_year",
    "PERIOD_BEGIN",
    "per_capita_income",
    "HOUSING_MARKET_INDEX",
    "HOUSING_MARKET_INDEX_MOM",
    "month_offset_from_incident",
    "incident_num",
]
CLUSTER_COLUMNS = [
    "median_ppsf_response_cluster",
    "median_ppsf_response_cluster_name",
    "median_ppsf_response_cluster_interpretation",
    "median_ppsf_response_cluster_algorithm",
    "median_ppsf_response_cluster_k",
    "median_ppsf_response_cluster_silhouette",
    "median_ppsf_response_incident_count",
]
COUNTY_SUMMARY_METRICS = {
    "HOUSING_MARKET_INDEX": "HOUSING_MARKET_INDEX_change_in_yoy_12_to_24",
    "AVG_SALE_TO_LIST": "AVG_SALE_TO_LIST_change_in_yoy_12_to_24",
    "HOMES_SOLD": "HOMES_SOLD_change_in_yoy_12_to_24",
    "INVENTORY": "INVENTORY_change_in_yoy_12_to_24",
    "MEDIAN_PPSF": "MEDIAN_PPSF_change_in_yoy_12_to_24",
}


def _existing_columns(df, columns):
    return [column for column in columns if column in df.columns]


def _write_csv_subset(df, path, columns):
    subset = df.loc[:, _existing_columns(df, columns)].copy()
    if "fips" in subset.columns:
        subset["fips"] = subset["fips"].astype(str).str.zfill(5)
    subset.to_csv(path, index = False)


def _is_county_fips(series):
    fips = series.astype(str).str.zfill(5)
    return fips.str.len().eq(5) & fips.str.slice(2).ne("000")


def _weighted_incident_average(incidents):
    if not incidents:
        return None, 0
    incidents = sorted(incidents, key = lambda item: item["incident_num"])
    total_weight = len(incidents) * (len(incidents) + 1) / 2
    value = sum((idx + 1) * item["value"] for idx, item in enumerate(incidents)) / total_weight
    return value, len(incidents)


def _build_yoy_summary_payload(df):
    payload = {"metrics": {}}
    if df.empty:
        return payload

    work = df.copy()
    work["fips"] = work["fips"].astype(str).str.zfill(5)
    work = work.loc[_is_county_fips(work["fips"])]
    work["incident_num"] = pd.to_numeric(work["incident_num"], errors = "coerce")
    work["month_offset_from_incident"] = pd.to_numeric(work["month_offset_from_incident"], errors = "coerce")
    work = work.dropna(subset = ["incident_num", "month_offset_from_incident"])

    period_configs = {
        "months_1_12": range(1, 13),
        "months_13_24": range(13, 25),
    }
    for metric in REQUESTED_HOUSING_METRICS:
        metric_col = f"{metric}_YOY_MOM"
        if metric_col not in work.columns:
            continue
        metric_payload = {}
        metric_values = pd.to_numeric(work[metric_col], errors = "coerce")
        metric_work = work.assign(_metric_value = metric_values).dropna(subset = ["_metric_value"])
        for period_key, required_offsets in period_configs.items():
            required_offsets = list(required_offsets)
            period_payload = {"all": {}, "complete": {}}
            incidents_by_county = {}
            complete_incidents_by_county = {}
            for (fips, incident_num), group in metric_work.groupby(["fips", "incident_num"], dropna = False):
                period_rows = group.loc[group["month_offset_from_incident"].isin(required_offsets)]
                if period_rows.empty:
                    continue
                by_offset = period_rows.groupby("month_offset_from_incident")["_metric_value"].agg(["mean", "count"])
                value = float(by_offset["mean"].mean())
                incident = {"incident_num": float(incident_num), "value": value}
                incidents_by_county.setdefault(fips, []).append(incident)
                has_complete_offsets = set(by_offset.index.astype(int)) == set(required_offsets)
                has_no_duplicates = bool((by_offset["count"] == 1).all())
                if has_complete_offsets and has_no_duplicates:
                    complete_incidents_by_county.setdefault(fips, []).append(incident)

            for fips, incidents in incidents_by_county.items():
                value, count = _weighted_incident_average(incidents)
                if value is not None:
                    period_payload["all"][fips] = {"value": value, "incidentCount": count}
            for fips, incidents in complete_incidents_by_county.items():
                value, count = _weighted_incident_average(incidents)
                if value is not None:
                    period_payload["complete"][fips] = {"value": value, "incidentCount": count}
            metric_payload[period_key] = period_payload
        payload["metrics"][metric] = metric_payload
    return payload


def _build_county_summary_df(df):
    fieldnames = ["fips", "county_name", "county_profile", *CLUSTER_COLUMNS]
    for metric in COUNTY_SUMMARY_METRICS:
        fieldnames.extend([
            f"{metric}_change_all",
            f"{metric}_incident_count_all",
            f"{metric}_change_complete",
            f"{metric}_incident_count_complete",
        ])

    if df.empty:
        return pd.DataFrame(columns = fieldnames)

    work = df.copy()
    work["fips"] = work["fips"].astype(str).str.zfill(5)
    work = work.loc[_is_county_fips(work["fips"])]
    work["incident_num"] = pd.to_numeric(work["incident_num"], errors = "coerce")
    work["month_offset_from_incident"] = pd.to_numeric(work["month_offset_from_incident"], errors = "coerce")
    work = work.dropna(subset = ["incident_num"])

    incident_summaries_by_county = {}
    for (fips, incident_num), group in work.groupby(["fips", "incident_num"], dropna = False):
        latest = group.iloc[-1]
        incident_summary = {
            "fips": fips,
            "incident_num": float(incident_num),
            "county_name": latest.get("REGION") or latest.get("county_name") or fips,
            "county_profile": latest.get("county_profile", pd.NA),
            "cluster_fields": {
                column: latest.get(column, "")
                for column in CLUSTER_COLUMNS
                if column in work.columns and pd.notna(latest.get(column, pd.NA)) and latest.get(column, "") != ""
            },
            "metrics": {},
        }
        offsets = set(pd.to_numeric(group["month_offset_from_incident"], errors = "coerce").dropna().astype(int))
        for metric, change_col in COUNTY_SUMMARY_METRICS.items():
            if change_col not in group.columns or metric not in group.columns:
                continue
            change_values = pd.to_numeric(group[change_col], errors = "coerce").dropna()
            if change_values.empty:
                continue
            metric_values = pd.to_numeric(group[metric], errors = "coerce")
            metric_offsets = set(group.loc[metric_values.notna(), "month_offset_from_incident"].dropna().astype(int))
            incident_summary["metrics"][metric] = {
                "value": float(change_values.iloc[0]),
                "complete": set(range(-12, 0)).issubset(metric_offsets) and set(range(1, 25)).issubset(metric_offsets),
            }
        incident_summaries_by_county.setdefault(fips, []).append(incident_summary)

    rows = []
    for fips, incidents in sorted(incident_summaries_by_county.items()):
        incidents = sorted(incidents, key = lambda item: item["incident_num"])
        representative = incidents[-1]
        row = {
            "fips": fips,
            "county_name": representative["county_name"],
            "county_profile": representative["county_profile"],
        }
        row.update(representative["cluster_fields"])
        has_any_metric = False
        for metric in COUNTY_SUMMARY_METRICS:
            all_incidents = [
                {"incident_num": incident["incident_num"], "value": incident["metrics"][metric]["value"]}
                for incident in incidents
                if metric in incident["metrics"]
            ]
            complete_incidents = [
                {"incident_num": incident["incident_num"], "value": incident["metrics"][metric]["value"]}
                for incident in incidents
                if metric in incident["metrics"] and incident["metrics"][metric]["complete"]
            ]
            all_value, all_count = _weighted_incident_average(all_incidents)
            complete_value, complete_count = _weighted_incident_average(complete_incidents)
            row[f"{metric}_change_all"] = "" if all_value is None else all_value
            row[f"{metric}_incident_count_all"] = all_count
            row[f"{metric}_change_complete"] = "" if complete_value is None else complete_value
            row[f"{metric}_incident_count_complete"] = complete_count
            has_any_metric = has_any_metric or all_count > 0 or complete_count > 0
        if has_any_metric:
            rows.append(row)
    return pd.DataFrame(rows, columns = fieldnames)


def _find_open_port(host = "127.0.0.1", start = 8000, end = 8100):
    for port in range(start, end + 1):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            if sock.connect_ex((host, port)) != 0:
                return port
    raise RuntimeError("Could not find an open port for the local visualization server.")


def _incident_type_slug(incident_type):
    slug = re.sub(r"[^a-z0-9]+", "_", str(incident_type).strip().lower())
    return slug.strip("_") or "unknown"


def _assign_three_way_change_bins(series):
    values = pd.to_numeric(series, errors = "coerce")
    result = pd.Series(pd.NA, index = series.index, dtype = "object")
    valid = values.dropna()
    if valid.empty:
        return result

    if valid.nunique() == 1:
        result.loc[valid.index] = YOY_12_TO_24_BIN_LABELS[1]
        return result

    try:
        codes = pd.qcut(valid, q = 3, labels = [0, 1, 2], duplicates = "drop")
    except ValueError:
        result.loc[valid.index] = YOY_12_TO_24_BIN_LABELS[1]
        return result

    if getattr(codes, "cat", None) is not None and len(codes.cat.categories) < 3:
        result.loc[valid.index] = YOY_12_TO_24_BIN_LABELS[1]
        return result

    result.loc[valid.index] = codes.astype("Int64").map(YOY_12_TO_24_BIN_LABELS)
    return result


def _add_change_in_yoy_12_to_24_columns(df):
    if df.empty:
        for metric in YOY_12_TO_24_METRICS:
            base_col = f"{metric}_change_in_yoy_12_to_24"
            df[base_col] = pd.Series(dtype = "float64")
            df[f"{base_col}_bin"] = pd.Series(dtype = "object")
        return df

    out = df.copy()
    group_cols = ["fips", "incident_num"]

    for metric, yoy_mom_col in YOY_12_TO_24_METRICS.items():
        base_col = f"{metric}_change_in_yoy_12_to_24"
        if yoy_mom_col not in out.columns:
            out[base_col] = pd.NA
            out[f"{base_col}_bin"] = pd.NA
            continue

        grouped = (
            out.groupby(group_cols, dropna = False)
            .apply(
                lambda group: pd.Series({
                    "months_1_12": pd.to_numeric(
                        group.loc[group["month_offset_from_incident"].between(1, 12), yoy_mom_col],
                        errors = "coerce",
                    ).mean(),
                    "months_13_24": pd.to_numeric(
                        group.loc[group["month_offset_from_incident"].between(13, 24), yoy_mom_col],
                        errors = "coerce",
                    ).mean(),
                })
            )
            .reset_index()
        )
        grouped[base_col] = grouped["months_13_24"] - grouped["months_1_12"]
        grouped[f"{base_col}_bin"] = _assign_three_way_change_bins(grouped[base_col])
        grouped = grouped[group_cols + [base_col, f"{base_col}_bin"]]
        out = out.merge(grouped, on = group_cols, how = "left")

    return out

def launch_bivariate_map(incident_type = None, months = 12, host = "127.0.0.1", port = None):
    output_dir = Path("..") / "output" / "visualizations"
    output_dir.mkdir(parents = True, exist_ok = True)

    prepared_incidents = _prepare_incident_df(incident_type = incident_type)
    incident_types = sorted(prepared_incidents["incidentType"].dropna().unique())

    incident_housing_dfs = {}
    incident_housing_24m_dfs = {}
    csv_paths = {}
    manifest_entries = []

    for current_incident_type in incident_types:
        housing_subset = get_housing_data_in_window(incident_type = current_incident_type, months = months)
        if isinstance(housing_subset, str):
            raise ValueError(housing_subset)

        housing_subset_24m = get_housing_data_in_window2(
            incident_type = current_incident_type,
            months_before = 12,
            months_after = 24,
        )
        housing_subset_24m = _add_change_in_yoy_12_to_24_columns(housing_subset_24m)
        housing_subset_24m = housing_subset_24m.drop(columns = YOY_12_TO_24_DROP_COLUMNS, errors = "ignore")

        cluster_annotations = globals().get("ppsf_response_cluster_annotations_df")
        if cluster_annotations is not None and not cluster_annotations.empty:
            cluster_columns = [
                "median_ppsf_response_cluster",
                "median_ppsf_response_cluster_name",
                "median_ppsf_response_cluster_interpretation",
                "median_ppsf_response_cluster_algorithm",
                "median_ppsf_response_cluster_k",
                "median_ppsf_response_cluster_silhouette",
                "median_ppsf_response_incident_count",
            ]
            current_cluster_annotations = cluster_annotations.loc[
                cluster_annotations["incident_type"] == current_incident_type,
                ["fips"] + cluster_columns,
            ].copy()
            if not current_cluster_annotations.empty:
                current_cluster_annotations["fips"] = current_cluster_annotations["fips"].astype(str).str.zfill(5)
                for frame_name, frame in [("housing_subset", housing_subset), ("housing_subset_24m", housing_subset_24m)]:
                    frame = frame.drop(columns = cluster_columns, errors = "ignore").copy()
                    frame["fips"] = frame["fips"].astype(str).str.zfill(5)
                    frame = frame.merge(current_cluster_annotations, on = "fips", how = "left")
                    if frame_name == "housing_subset":
                        housing_subset = frame
                    else:
                        housing_subset_24m = frame

        incident_slug = _incident_type_slug(current_incident_type)
        index_housing_csv_path = output_dir / f"{incident_slug}_index_housing.csv"
        index_housing_24m_csv_path = output_dir / f"{incident_slug}_index_housing_24mths.csv"
        story_1_csv_path = output_dir / f"{incident_slug}_story_1_housing.csv"
        story_2_csv_path = output_dir / f"{incident_slug}_story_2_housing_24mths.csv"
        county_summary_csv_path = output_dir / f"{incident_slug}_county_summary.csv"
        index_yoy_summary_path = output_dir / f"{incident_slug}_index_housing_24mths_yoy_summary.json"

        _write_csv_subset(housing_subset, index_housing_csv_path, INDEX_HOUSING_COLUMNS)
        _write_csv_subset(housing_subset_24m, index_housing_24m_csv_path, INDEX_24M_COLUMNS)
        _write_csv_subset(housing_subset, story_1_csv_path, STORY_1_COLUMNS)
        _write_csv_subset(housing_subset_24m, story_2_csv_path, STORY_2_COLUMNS)
        _build_county_summary_df(housing_subset_24m).to_csv(county_summary_csv_path, index = False)
        index_yoy_summary_path.write_text(
            json.dumps(_build_yoy_summary_payload(housing_subset_24m), separators = (",", ":")),
            encoding = "utf-8",
        )

        incident_housing_dfs[current_incident_type] = housing_subset
        incident_housing_24m_dfs[current_incident_type] = housing_subset_24m
        csv_paths[current_incident_type] = {
            "index_housing": index_housing_csv_path,
            "index_housing_24mths": index_housing_24m_csv_path,
            "story_1_housing": story_1_csv_path,
            "story_2_housing_24mths": story_2_csv_path,
            "county_summary": county_summary_csv_path,
            "index_yoy_summary": index_yoy_summary_path,
        }
        manifest_entry = {
            "incident_type": current_incident_type,
            "slug": incident_slug,
            "index_housing_csv": index_housing_csv_path.name,
            "index_housing_24mths_csv": index_housing_24m_csv_path.name,
            "index_yoy_summary_json": index_yoy_summary_path.name,
            "story_1_housing_csv": story_1_csv_path.name,
            "story_2_housing_24mths_csv": story_2_csv_path.name,
            "county_summary_csv": county_summary_csv_path.name,
            "housing_csv": index_housing_csv_path.name,
            "housing_24mths_csv": index_housing_24m_csv_path.name,
        }
        if globals().get("ppsf_response_cluster_interpretations_df") is not None:
            manifest_entry["ppsf_response_cluster_summary_json"] = "ppsf_response_cluster_summaries.json"
        manifest_entries.append(manifest_entry)

    default_incident_type = "Fire" if "Fire" in incident_types else (incident_types[0] if incident_types else None)
    manifest = {
        "default_incident_type": default_incident_type,
        "incident_types": manifest_entries,
    }
    cluster_interpretations = globals().get("ppsf_response_cluster_interpretations_df")
    if cluster_interpretations is not None and not cluster_interpretations.empty:
        cluster_summaries = {}
        for incident, group in cluster_interpretations.groupby("incident_type"):
            first_row = group.iloc[0]
            cluster_summaries[incident] = {
                "incident_type": incident,
                "status": "clustered",
                "algorithm": first_row["algorithm"],
                "k": int(first_row["k"]),
                "silhouette_score": float(first_row["silhouette_score"]),
                "metrics_used": first_row.get("metrics_used", ", ".join(PPSF_RESPONSE_METRIC_LABELS.values())),
                "counties_clustered": int(group["counties"].sum()),
                "clusters": [
                    {
                        "cluster": int(row.cluster),
                        "name": row.cluster_name,
                        "interpretation": row.interpretation,
                        "counties": int(row.counties),
                    }
                    for row in group.sort_values("cluster").itertuples(index = False)
                ],
            }
        cluster_summary_path = output_dir / "ppsf_response_cluster_summaries.json"
        cluster_summary_path.write_text(json.dumps(cluster_summaries, indent = 2), encoding = "utf-8")

    manifest_path = output_dir / "incident_housing_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent = 2), encoding = "utf-8")


    index_html_path = output_dir / "index.html"
    climate_on_housing_path = output_dir / "climate-on-housing.html"
    if not index_html_path.exists():
        raise FileNotFoundError(f"Visualization file not found: {index_html_path.resolve()}")
    if not climate_on_housing_path.exists():
        raise FileNotFoundError(f"Visualization file not found: {climate_on_housing_path.resolve()}")

    existing_server = globals().get("_bivariate_map_server")
    if existing_server is not None:
        existing_server.shutdown()
        existing_server.server_close()

    selected_port = port or _find_open_port(host = host)
    handler = partial(SimpleHTTPRequestHandler, directory = str(output_dir.resolve()))
    httpd = ThreadingHTTPServer((host, selected_port), handler)
    server_thread = threading.Thread(target = httpd.serve_forever, daemon = True)
    server_thread.start()

    map_url = f"http://{host}:{selected_port}/index.html"
    complete_case_map_url = f"{map_url}?complete_cases=1"
    climate_on_housing_url = f"http://{host}:{selected_port}/climate-on-housing.html"

    globals()["_bivariate_map_server"] = httpd
    globals()["_bivariate_map_url"] = map_url
    globals()["_bivariate_map_complete_case_url"] = complete_case_map_url
    globals()["_climate_on_housing_url"] = climate_on_housing_url
    globals()["_incident_housing_dfs"] = incident_housing_dfs
    globals()["_incident_housing_24m_dfs"] = incident_housing_24m_dfs
    globals()["_incident_housing_csv_paths"] = csv_paths
    globals()["_incident_housing_manifest"] = manifest

    for current_incident_type in incident_types:
        print(f"Saved {current_incident_type} index housing data to: {csv_paths[current_incident_type]['index_housing'].resolve()}")
        print(f"Saved {current_incident_type} index 24-month housing data to: {csv_paths[current_incident_type]['index_housing_24mths'].resolve()}")
    print(f"Saved incident manifest to: {manifest_path.resolve()}")
    print(f"Open the bivariate map at: {map_url}")
    print(f"Open the complete-case bivariate map at: {complete_case_map_url}")
    print(f"Open the climate-on-housing page at: {climate_on_housing_url}")

    return incident_housing_dfs, {
        "default": map_url,
        "complete_case": complete_case_map_url,
        "climate_on_housing": climate_on_housing_url,
        "csv_paths": csv_paths,
        "manifest": manifest,
    }

incident_housing_dfs, bivariate_map_urls = launch_bivariate_map(incident_type = None, months = 12)



Saved Coastal Storm index housing data to: C:\Users\kenny\Data Science Projects\quoll-intelligence\output\visualizations\coastal_storm_index_housing.csv
Saved Coastal Storm index 24-month housing data to: C:\Users\kenny\Data Science Projects\quoll-intelligence\output\visualizations\coastal_storm_index_housing_24mths.csv
Saved Earthquake index housing data to: C:\Users\kenny\Data Science Projects\quoll-intelligence\output\visualizations\earthquake_index_housing.csv
Saved Earthquake index 24-month housing data to: C:\Users\kenny\Data Science Projects\quoll-intelligence\output\visualizations\earthquake_index_housing_24mths.csv
Saved Fire index housing data to: C:\Users\kenny\Data Science Projects\quoll-intelligence\output\visualizations\fire_index_housing.csv
Saved Fire index 24-month housing data to: C:\Users\kenny\Data Science Projects\quoll-intelligence\output\visualizations\fire_index_housing_24mths.csv
Saved Flood index housing data to: C:\Users\kenny\Data Science Projects\quoll-inte

## Statistical Tests

Statistical tests to determine if the values before and after the incident are significantly different.

In [34]:
# Difference in means of 6-month-before and 6-month-after time periods


In [35]:
# Linear Mixed Model
import statsmodels.formula.api as smf
